# Stock Price Prediction Using Historical Data & Market News

**Objective**: Predict next-day closing prices for 7 tech mega-caps (AAPL, AMZN, GOOGL, META, MSFT, NVDA, TSLA) by combining technical indicators from OHLCV data with NLP-derived sentiment features from financial news.

**Data**: ~241 trading days per ticker (Nov 2023 -- Oct 2024), 4,440 news articles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = '.'

## 1. Data Loading

In [ ]:
price_df = pd.read_csv(f'{DATA_DIR}/price.csv', parse_dates=['date'])
news_df = pd.read_csv(f'{DATA_DIR}/news.csv', parse_dates=['datetime'])

print(f'Price shape: {price_df.shape}')
print(f'News shape:  {news_df.shape}')
print()
print('--- Price ---')
display(price_df.head())
display(price_df.info())
print()
print('--- News ---')
display(news_df.head())
display(news_df.info())

## 2. Price Data -- EDA

In [ ]:
print('=== Basic Statistics ===')
display(price_df.describe())

print('\n=== Missing Values ===')
display(price_df.isnull().sum())

print('\n=== Tickers ===')
tickers = sorted(price_df['ticker'].unique())
print(tickers)
print(f'Trading days per ticker: {price_df.groupby("ticker").size().unique()}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharey=False)
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    sub = price_df[price_df['ticker'] == ticker]
    axes[i].plot(sub['date'], sub['close'], linewidth=1.2)
    axes[i].set_title(ticker, fontsize=13, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].set_ylabel('Close ($)')

axes[-1].set_visible(False)
fig.suptitle('Closing Prices (Nov 2023 -- Oct 2024)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
price_df = price_df.sort_values(['ticker', 'date']).reset_index(drop=True)
price_df['return_1d'] = price_df.groupby('ticker')['close'].pct_change()
price_df['log_return_1d'] = np.log(price_df['close'] / price_df.groupby('ticker')['close'].shift(1))

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    sub = price_df[price_df['ticker'] == ticker]
    axes[i].hist(sub['return_1d'].dropna(), bins=40, edgecolor='white', alpha=0.85)
    axes[i].axvline(0, color='red', linestyle='--', alpha=0.5)
    mu = sub['return_1d'].mean()
    sigma = sub['return_1d'].std()
    axes[i].set_title(f'{ticker} (mu={mu:.4f}, std={sigma:.4f})', fontsize=11)
    axes[i].set_xlabel('Daily Return')

axes[-1].set_visible(False)
fig.suptitle('Distribution of Daily Returns', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
returns_pivot = price_df.pivot_table(index='date', columns='ticker', values='return_1d')

fig, ax = plt.subplots(figsize=(8, 7))
corr = returns_pivot.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, square=True, ax=ax)
ax.set_title('Cross-Ticker Daily Return Correlation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key takeaway: High correlations suggest cross-ticker features (market momentum) will be informative.')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    sub = price_df[price_df['ticker'] == ticker]
    axes[i].bar(sub['date'], sub['volume'], width=1, alpha=0.7)
    axes[i].set_title(ticker, fontsize=13, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].set_ylabel('Volume')

axes[-1].set_visible(False)
fig.suptitle('Trading Volume Over Time', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('=== Augmented Dickey-Fuller Test ===')
print(f'{"Ticker":<8} {"Close p-val":>12} {"Stationary?":>12} {"Return p-val":>13} {"Stationary?":>12}')
print('-' * 62)

for ticker in tickers:
    sub = price_df[price_df['ticker'] == ticker].dropna()
    adf_close = adfuller(sub['close'], autolag='AIC')
    adf_ret = adfuller(sub['return_1d'].dropna(), autolag='AIC')
    stat_close = 'Yes' if adf_close[1] < 0.05 else 'No'
    stat_ret = 'Yes' if adf_ret[1] < 0.05 else 'No'
    print(f'{ticker:<8} {adf_close[1]:>12.4f} {stat_close:>12} {adf_ret[1]:>13.4f} {stat_ret:>12}')

print('\nNote: Close prices are typically non-stationary; returns are stationary.')
print('This supports using returns/changes as features rather than raw prices.')

## 3. News Data -- EDA

In [ ]:
print('=== News Basic Stats ===')
print(f'Total articles: {len(news_df)}')
print(f'Date range: {news_df["datetime"].min()} to {news_df["datetime"].max()}')
print(f'\nArticles per ticker:')
display(news_df['ticker'].value_counts().sort_index())

print(f'\nMissing values:')
display(news_df.isnull().sum())

print(f'\nHeadline length stats: mean={news_df["headline"].str.len().mean():.0f}, '
      f'median={news_df["headline"].str.len().median():.0f}')
print(f'Summary length stats:  mean={news_df["summary"].str.len().mean():.0f}, '
      f'median={news_df["summary"].str.len().median():.0f}')

In [ ]:
news_df['date'] = news_df['datetime'].dt.date
news_daily = news_df.groupby(['date', 'ticker']).size().reset_index(name='news_count')
news_daily['date'] = pd.to_datetime(news_daily['date'])

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    sub = news_daily[news_daily['ticker'] == ticker]
    axes[i].bar(sub['date'], sub['news_count'], width=1, alpha=0.7, color='coral')
    axes[i].set_title(f'{ticker} ({sub["news_count"].sum()} articles)', fontsize=11)
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].set_ylabel('# Articles')

axes[-1].set_visible(False)
fig.suptitle('Daily News Volume Per Ticker', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
trading_dates = price_df['date'].dt.date.unique()
news_dates_per_ticker = news_df.groupby('ticker')['date'].apply(set)

print('=== News Coverage vs Trading Days ===')
print(f'{"Ticker":<8} {"Trading Days":>13} {"Days w/ News":>13} {"Coverage %":>11} {"Days Missing":>13}')
print('-' * 62)
for ticker in tickers:
    td = set(price_df[price_df['ticker'] == ticker]['date'].dt.date.unique())
    nd = news_dates_per_ticker.get(ticker, set())
    covered = len(td & nd)
    print(f'{ticker:<8} {len(td):>13} {covered:>13} {100*covered/len(td):>10.1f}% {len(td)-covered:>13}')

print('\nNote: GOOGL and NVDA have significant gaps -- will need neutral fill strategy.')

In [ ]:
news_df['hour'] = news_df['datetime'].dt.hour

fig, ax = plt.subplots(figsize=(12, 5))
news_df['hour'].hist(bins=24, range=(0, 24), edgecolor='white', alpha=0.85, ax=ax)
ax.axvspan(9.5, 16, alpha=0.15, color='green', label='Market hours (9:30-16:00 ET)')
ax.axvline(16, color='red', linestyle='--', alpha=0.7, label='Market close (16:00 ET)')
ax.set_xlabel('Hour of Day (assuming ET)', fontsize=12)
ax.set_ylabel('Number of Articles', fontsize=12)
ax.set_title('News Publication Hour Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(range(0, 25))
plt.tight_layout()
plt.show()

after_close = (news_df['hour'] >= 16).sum()
before_open = (news_df['hour'] < 9).sum()
during_market = len(news_df) - after_close - before_open
print(f'Before market open (<9:30): {before_open} ({100*before_open/len(news_df):.1f}%)')
print(f'During market hours:        {during_market} ({100*during_market/len(news_df):.1f}%)')
print(f'After market close (>=16):  {after_close} ({100*after_close/len(news_df):.1f}%)')
print('\nStrategy: News after 16:00 will be assigned to the NEXT trading day\'s features.')

In [ ]:
print('=== Sample Headlines Per Ticker ===')
for ticker in tickers:
    sub = news_df[news_df['ticker'] == ticker].head(3)
    print(f'\n--- {ticker} ---')
    for _, row in sub.iterrows():
        print(f'  [{row["datetime"].strftime("%Y-%m-%d %H:%M")}] {row["headline"][:100]}')

In [ ]:
price_df['abs_return'] = price_df['return_1d'].abs()
price_daily = price_df[['date', 'ticker', 'abs_return']].copy()
price_daily['date_key'] = price_daily['date'].dt.date

news_count_daily = news_df.groupby(['date', 'ticker']).size().reset_index(name='news_count')
news_count_daily.rename(columns={'date': 'date_key'}, inplace=True)

merged = price_daily.merge(news_count_daily, on=['date_key', 'ticker'], how='left')
merged['news_count'] = merged['news_count'].fillna(0)

fig, ax = plt.subplots(figsize=(10, 6))
for ticker in tickers:
    sub = merged[merged['ticker'] == ticker]
    ax.scatter(sub['news_count'], sub['abs_return'], alpha=0.4, s=20, label=ticker)

ax.set_xlabel('Daily News Count', fontsize=12)
ax.set_ylabel('Absolute Daily Return', fontsize=12)
ax.set_title('News Volume vs Price Volatility', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

overall_corr = merged[['news_count', 'abs_return']].corr().iloc[0, 1]
print(f'Overall correlation (news_count vs abs_return): {overall_corr:.3f}')

## EDA Summary

**Key Findings:**

1. **Price data** is clean (no missing values), covers 241 trading days per ticker. Close prices are non-stationary but returns are stationary -- will use returns/changes as features.
2. **Cross-ticker correlations** are moderate to high, confirming that market-level features will add value.
3. **News coverage** varies significantly: AMZN has the most articles (941), GOOGL the fewest (386). Several tickers have 20-35% of trading days without any news -- neutral fill strategy is required.
4. **Temporal alignment** matters: a meaningful fraction of news is published outside market hours. Post-close news will be attributed to the next trading day.
5. **News volume vs volatility**: moderate positive correlation, supporting news_count as a useful feature.